<a href="https://colab.research.google.com/github/safaabuzaid/mri-generalization/blob/main/data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
from collections import defaultdict
import hashlib
import pandas as pd

from PIL import Image
from sklearn.model_selection import train_test_split

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
DATA_A_DIR = Path("/content/drive/MyDrive/MRI_Generalization/project_data/dataA_figshare/extracted_images")

DATA_B_train = Path("/content/drive/MyDrive/MRI_Generalization/project_data/SARTAJ/Training")
DATA_B_test = Path("/content/drive/MyDrive/MRI_Generalization/project_data/SARTAJ/Testing")

OUTPUT_DIR = Path("/content/drive/MyDrive/MRI_Generalization/project_data/splits")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

CLASS_NAMES = ["glioma", "meningioma", "pituitary"]

LABEL_MAP = {
    "glioma": 0,
    "meningioma": 1,
    "pituitary": 2
}

In [5]:
def get_image_hash(path):
    """
    Create an MD5 hash from the actual pixel data.
    Images with identical pixels will have the same hash.
    """
    with Image.open(path) as img:
        img = img.convert("RGB")
        return hashlib.md5(img.tobytes()).hexdigest()

In [6]:
def collect_images(folder):
    records = []

    for class_dir in sorted(folder.iterdir()):

        if not class_dir.is_dir():
            continue

        class_name = class_dir.name.lower()

        if class_name not in LABEL_MAP:
            print(f"Warning: Unknown class found: {class_name}")
            continue

        for path in class_dir.rglob("*"):

            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                records.append({
                    "path": str(path),
                    "class_name": class_name,
                    "label": LABEL_MAP[class_name]
                })

    return pd.DataFrame(records)

In [7]:
dataA_df = collect_images(DATA_A_DIR)

print("DataA images:", len(dataA_df))
print("\nClass distribution:")
print(dataA_df["class_name"].value_counts())

DataA images: 3064

Class distribution:
class_name
glioma        1426
pituitary      930
meningioma     708
Name: count, dtype: int64


In [8]:
dataA_df["hash"] = dataA_df["path"].apply(get_image_hash)

dataA_duplicate_groups = (
    dataA_df.groupby("hash")
    .filter(lambda x: len(x) > 1)
)

print(
    "DataA duplicate groups:",
    dataA_duplicate_groups["hash"].nunique()
)

DataA duplicate groups: 0


In [10]:
dataB1_df = collect_images(DATA_B_train)
dataB2_df = collect_images(DATA_B_test)

print("DataB Folder 1:", len(dataB1_df))
print("DataB Folder 2:", len(dataB2_df))

print("\nFolder 1 distribution:")
print(dataB1_df["class_name"].value_counts())

print("\nFolder 2 distribution:")
print(dataB2_df["class_name"].value_counts())

DataB Folder 1: 2475
DataB Folder 2: 289

Folder 1 distribution:
class_name
pituitary     827
glioma        826
meningioma    822
Name: count, dtype: int64

Folder 2 distribution:
class_name
meningioma    115
glioma        100
pituitary      74
Name: count, dtype: int64


In [11]:
dataB1_df["hash"] = dataB1_df["path"].apply(get_image_hash)
dataB2_df["hash"] = dataB2_df["path"].apply(get_image_hash)

In [12]:
dup_B1 = (
    dataB1_df.groupby("hash")
    .filter(lambda x: len(x) > 1)
)

dup_B2 = (
    dataB2_df.groupby("hash")
    .filter(lambda x: len(x) > 1)
)

print("Duplicate groups inside Folder 1:",
      dup_B1["hash"].nunique())

print("Duplicate groups inside Folder 2:",
      dup_B2["hash"].nunique())

Duplicate groups inside Folder 1: 29
Duplicate groups inside Folder 2: 41


In [13]:
B1_hashes = set(dataB1_df["hash"])
B2_hashes = set(dataB2_df["hash"])

cross_B_duplicates = B1_hashes.intersection(B2_hashes)

print(
    "Duplicate groups across Folder 1 ↔ Folder 2:",
    len(cross_B_duplicates)
)

Duplicate groups across Folder 1 ↔ Folder 2: 132


In [14]:
different_label_duplicates = []

for h in cross_B_duplicates:

    rows1 = dataB1_df[dataB1_df["hash"] == h]
    rows2 = dataB2_df[dataB2_df["hash"] == h]

    labels1 = set(rows1["label"])
    labels2 = set(rows2["label"])

    if labels1 != labels2:
        different_label_duplicates.append(h)

print(
    "Cross-folder duplicates with different labels:",
    len(different_label_duplicates)
)

Cross-folder duplicates with different labels: 0


In [15]:
dataA_hashes = set(dataA_df["hash"])
dataB_hashes = B1_hashes.union(B2_hashes)

A_B_overlap = dataA_hashes.intersection(dataB_hashes)

print(
    "Exact duplicate groups between DataA and DataB:",
    len(A_B_overlap)
)

Exact duplicate groups between DataA and DataB: 0


In [16]:
dataB_df = pd.concat(
    [dataB1_df, dataB2_df],
    ignore_index=True
)

print("Raw DataB images:", len(dataB_df))

Raw DataB images: 2764


In [17]:
dataB_clean_df = dataB_df.drop_duplicates(
    subset="hash",
    keep="first"
).copy()

dataB_clean_df = dataB_clean_df.reset_index(drop=True)

print("Raw DataB images:", len(dataB_df))
print("Clean DataB images:", len(dataB_clean_df))
print("Removed:", len(dataB_df) - len(dataB_clean_df))

Raw DataB images: 2764
Clean DataB images: 2543
Removed: 221


In [18]:
print(dataB_clean_df["class_name"].value_counts())

print("\nPercentages:")
print(
    dataB_clean_df["class_name"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

class_name
glioma        886
pituitary     834
meningioma    823
Name: count, dtype: int64

Percentages:
class_name
glioma        34.84
pituitary     32.80
meningioma    32.36
Name: proportion, dtype: float64


In [19]:
dataA_df = dataA_df.drop(columns=["hash"])
dataB_clean_df = dataB_clean_df.drop(columns=["hash"])

In [20]:
train_df, val_df = train_test_split(
    dataA_df,
    test_size=0.20,  # 80% train, 20% validation
    stratify=dataA_df["label"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))

Train: 2451
Validation: 613


In [21]:
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(dataB_clean_df))

Train: 2451
Validation: 613
Test: 2543


In [22]:
print("TRAIN")
print(train_df["class_name"].value_counts())

print("\nVALIDATION")
print(val_df["class_name"].value_counts())

print("\nTEST")
print(dataB_clean_df["class_name"].value_counts())

TRAIN
class_name
glioma        1141
pituitary      744
meningioma     566
Name: count, dtype: int64

VALIDATION
class_name
glioma        285
pituitary     186
meningioma    142
Name: count, dtype: int64

TEST
class_name
glioma        886
pituitary     834
meningioma    823
Name: count, dtype: int64


In [23]:
train_df.to_csv(
    OUTPUT_DIR / "train_split.csv",
    index=False
)

val_df.to_csv(
    OUTPUT_DIR / "val_split.csv",
    index=False
)

dataB_clean_df.to_csv(
    OUTPUT_DIR / "test_split.csv",
    index=False
)

print("Splits saved successfully.")

Splits saved successfully.


In [24]:
print("=" * 50)
print("FINAL DATASET SUMMARY")
print("=" * 50)

print(f"Train images:      {len(train_df)}")
print(f"Validation images: {len(val_df)}")
print(f"Test images:       {len(dataB_clean_df)}")

print("\nLeakage checks:")
print(f"DataA internal duplicates: {dataA_duplicate_groups['hash'].nunique()}")
print(f"DataA ↔ DataB overlap:     {len(A_B_overlap)}")

print("\nDataB duplicates:")
print(f"Folder 1 internal:         {dup_B1['hash'].nunique()}")
print(f"Folder 2 internal:         {dup_B2['hash'].nunique()}")
print(f"Cross-folder:              {len(cross_B_duplicates)}")

print("\nSaved files:")
print(OUTPUT_DIR / "train_split.csv")
print(OUTPUT_DIR / "val_split.csv")
print(OUTPUT_DIR / "test_split.csv")

FINAL DATASET SUMMARY
Train images:      2451
Validation images: 613
Test images:       2543

Leakage checks:
DataA internal duplicates: 0
DataA ↔ DataB overlap:     0

DataB duplicates:
Folder 1 internal:         29
Folder 2 internal:         41
Cross-folder:              132

Saved files:
/content/drive/MyDrive/MRI_Generalization/project_data/splits/train_split.csv
/content/drive/MyDrive/MRI_Generalization/project_data/splits/val_split.csv
/content/drive/MyDrive/MRI_Generalization/project_data/splits/test_split.csv
